# Spatial Detector Comparison

**Detectors**: DSM · LinearDSM · NeighborMLP · AMF · AMF-local · GMM-Levin

- All spatial models use the **same k×k window** with **circular padding**.
- Targets are **never planted within `edge_guard` px** of the box edge.
- Best-epoch model is used for inference (lowest training loss across all epochs).

**Cells to run:**
1. `[Setup]` — clone / pull repo, mount Drive, install deps
2. `[Config]` — set all hyperparams
3. `[Reload]` — reload modules after any code change (no restart needed)
4. `[Run]` — train models → detect → save results to Drive
5. `[Show]` — display all figures from any result folder


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 1 — Setup: clone/pull repo, mount Drive, install deps
# ══════════════════════════════════════════════════════════════════════════
import os, sys, subprocess

REPO_URL  = 'https://github.com/michaelpiro/final-paper-experiment.git'
BRANCH    = 'iid-unified-experiment'
PROJ_DIR  = '/content/proj'

if os.path.exists(os.path.join(PROJ_DIR, '.git')):
    r = subprocess.run(['git', '-C', PROJ_DIR, 'pull', 'origin', BRANCH],
                       capture_output=True, text=True)
    print(r.stdout.strip()); print(r.stderr.strip())
else:
    subprocess.run(['git', 'clone', '-b', BRANCH, '--depth', '1',
                    REPO_URL, PROJ_DIR], check=True)

# Install Python deps (idempotent)
subprocess.run(['pip', 'install', '-q', 'scikit-learn', 'scipy', 'tqdm',
                'matplotlib', 'pyyaml'], check=True)

for p in [PROJ_DIR, os.path.join(PROJ_DIR, 'experiments', 'spatial')]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(PROJ_DIR)
print('CWD:', os.getcwd())

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Symlink dataset (edit path if needed)
os.makedirs(f'{PROJ_DIR}/real_datasets', exist_ok=True)
DATA_DST = f'{PROJ_DIR}/real_datasets/pavia-u.mat'
if not os.path.exists(DATA_DST):
    candidates = [
        '/content/drive/MyDrive/final_paper/pavia-u.mat',
        '/content/drive/MyDrive/final_paper/real_datasets/pavia-u.mat',
    ]
    for src in candidates:
        if os.path.exists(src):
            os.symlink(src, DATA_DST)
            print('Dataset linked:', src)
            break
    else:
        raise FileNotFoundError(
            f'pavia-u.mat not found. Upload it to one of:\n  ' +
            '\n  '.join(candidates))
else:
    print('Dataset already present:', DATA_DST)

# Verify the edge_guard fix is in place
r = subprocess.run(['grep', '-c', 'edge_guard',
                    f'{PROJ_DIR}/final_paper_experiments/data_utils.py'],
                   capture_output=True, text=True)
print(f'edge_guard in data_utils.py: {r.stdout.strip()} occurrences (expect ≥ 3)')

r = subprocess.run(['git', '-C', PROJ_DIR, 'log', '--oneline', '-3'],
                   capture_output=True, text=True)
print('Last 3 commits:\n' + r.stdout)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 2 — Config
# Edit OVERRIDES; re-run this cell and then [Run] to launch with new params.
# ══════════════════════════════════════════════════════════════════════════

DRIVE_RESULTS = '/content/drive/MyDrive/final_paper/spatial_results'

OVERRIDES = dict(
    dataset           = f'{PROJ_DIR}/real_datasets/pavia-u.mat',
    results_dir       = DRIVE_RESULTS,
    manual_boxes_path = f'{PROJ_DIR}/experiments/spatial/manual_boxes.json',

    # ── scenario ─────────────────────────────────────────────────────────
    scenario_index    = 0,          # 0..3 = manual; 4..7 = random
    norm_mode         = 'none',     # raw bands; ZCA whitening inside nets

    # ── target planting ──────────────────────────────────────────────────
    amplitude         = 0.15,
    target_fraction   = 0.10,
    edge_guard        = 5,          # skip pixels ≤5 px from box edge

    # ── spatial window (same k for ALL spatial detectors) ─────────────────
    k                 = 5,          # k×k neighborhood; circular padding

    # ── DSM (global, nonlinear MLP) ───────────────────────────────────────
    dsm_hidden        = [128, 128],
    dsm_epochs        = 1000,
    dsm_lr            = 5e-4,

    # ── LinearDSM (neighbor-adapted ridge, paper §6.1) ────────────────────
    ridge_M           = 256,        # encoder output dim
    ridge_hidden      = [128, 128], # encoder hidden
    ridge_lam_init    = 0.1,        # ridge λ init (learned)
    ridge_epochs      = 1000,
    ridge_lr          = 3e-4,
    ridge_batch       = 256,
    ridge_n_mc        = 8,          # MC noise draws at scoring time

    # ── NeighborMLP (neighbor-adapted nonlinear denoiser) ─────────────────
    nmlp_d_lat        = 16,         # latent dim
    nmlp_K            = 8,          # top-K neighbor selection
    nmlp_enc_hidden   = [128, 64],  # encoder: D→128→64→d_lat
    nmlp_score_hidden = [128],      # score net: (D+(K+1)*d_lat)→128→D
    nmlp_epochs       = 1000,
    nmlp_lr           = 3e-4,
    nmlp_batch        = 256,

    # ── GMM-Levin ─────────────────────────────────────────────────────────
    gmm_steps         = 50,
    gmm_K             = 3,

    # ── detection / evaluation ────────────────────────────────────────────
    pfa_target        = 0.05,

    # ── shared ────────────────────────────────────────────────────────────
    activation        = 'silu',
    dsm_sigma_rho     = 0.01,
    whiten_mode       = 'zca',
    whiten_eig_floor  = 1e-5,
    batch_size        = 256,
    weight_decay      = 1e-4,
    local_scm_loading = 1e-8,
    baseline_eig_floor= 1e-12,
    seed              = 42,
)

import os; os.makedirs(DRIVE_RESULTS, exist_ok=True)
print('Config ready. Output dir:', DRIVE_RESULTS)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 3 — Reload modules  (run after any code change; no runtime restart)
# ══════════════════════════════════════════════════════════════════════════
import sys, importlib

# Ensure paths are set
for p in [PROJ_DIR, f'{PROJ_DIR}/experiments/spatial']:
    if p not in sys.path:
        sys.path.insert(0, p)

# Reload leaf → root order to avoid stale cross-references
_RELOAD_ORDER = [
    'final_paper_experiments.data_utils',
    'final_paper_experiments.baselines.detectors',
    'final_paper_experiments.baselines.gmm_glrt_levin',
    'final_paper_experiments.evaluation',
    'final_paper_experiments.models.neighbor_adapted',
    'dsm_model',
    'neighbor_mlp_model',
    'local_detectors',
    'run_colab',
    'run_compare',
]
for m in _RELOAD_ORDER:
    if m in sys.modules:
        importlib.reload(sys.modules[m])

import run_compare as rc
importlib.reload(rc)
print('Modules reloaded.')
print('DET_ORDER:', rc.DET_ORDER)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 4 — TRAIN → DETECT → SAVE
#
# Trains DSM, LinearDSM, NeighborMLP with best-epoch checkpointing.
# Runs detection (in-patch + foreign signatures).
# Saves models.pt + all figures + metrics to Drive.
# ══════════════════════════════════════════════════════════════════════════
import run_compare as rc, importlib; importlib.reload(rc)

RUN_DIR = rc.run_from_cfg(OVERRIDES, dry_run=False)
print('\n✓ Done. Results at:', RUN_DIR)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 4b — (Optional) Run multiple scenarios
# ══════════════════════════════════════════════════════════════════════════
# import run_compare as rc, importlib; importlib.reload(rc)
#
# RUN_DIRS = []
# for sidx in range(4):
#     ov = dict(OVERRIDES)
#     ov['scenario_index'] = sidx
#     rd = rc.run_from_cfg(ov)
#     RUN_DIRS.append(rd)
#     print(f'\n── Scenario {sidx} done → {rd}')
# RUN_DIR = RUN_DIRS[-1]
print('Uncomment the code above to run multiple scenarios.')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 5a — Show in-patch results
#
# RUN_DIR is set by Cell 4. You can also hardcode a path to an old run:
# RUN_DIR = '/content/drive/MyDrive/final_paper/spatial_results/compare_20260611_123456'
# ══════════════════════════════════════════════════════════════════════════
import run_compare as rc, importlib; importlib.reload(rc)

rc.show_plots_from_dir(RUN_DIR)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 5b — Show foreign-signature results
# ══════════════════════════════════════════════════════════════════════════
import run_compare as rc, importlib; importlib.reload(rc)

rc.show_plots_from_dir(RUN_DIR, sub='foreign')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 6 — Show plots from ANY saved folder (standalone)
#
# Set PLOT_DIR to any timestamped result folder — does not need a live
# RUN_DIR variable. Load figures from disk and display inline.
# ══════════════════════════════════════════════════════════════════════════
import run_compare as rc, importlib; importlib.reload(rc)

PLOT_DIR = RUN_DIR   # ← change to any path
# PLOT_DIR = '/content/drive/MyDrive/final_paper/spatial_results/compare_20260611_083635'

print('═' * 60)
print('In-patch signature')
print('═' * 60)
rc.show_plots_from_dir(PLOT_DIR)

print('═' * 60)
print('Foreign signature')
print('═' * 60)
rc.show_plots_from_dir(PLOT_DIR, sub='foreign')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 7 — Re-extract plots from a saved models.pt  (no retraining)
#
# Use this when you want to change SCORING params (pfa_target, amplitude,
# edge_guard, gmm_K) without retraining.
# Architecture keys (dsm_hidden, nmlp_*, ridge_*) are always taken from
# the saved checkpoint and cannot be overridden here.
# ══════════════════════════════════════════════════════════════════════════
import subprocess, sys, yaml, tempfile, os

MODELS_PT = f'{RUN_DIR}/models.pt'   # ← path to an existing models.pt

SCORING_OVERRIDES = dict(
    results_dir       = DRIVE_RESULTS,
    dataset           = OVERRIDES['dataset'],
    manual_boxes_path = OVERRIDES['manual_boxes_path'],
    pfa_target        = 0.05,
    amplitude         = 0.15,
    target_fraction   = 0.10,
    edge_guard        = 5,
    gmm_K             = 3,
    gmm_steps         = 50,
    ridge_n_mc        = 8,
    local_scm_loading = 1e-8,
    baseline_eig_floor= 1e-12,
)

# Write temp config
tmp = tempfile.NamedTemporaryFile(suffix='.yaml', delete=False, mode='w')
yaml.dump(SCORING_OVERRIDES, tmp); tmp.close()

result = subprocess.run(
    [sys.executable, '-u',
     f'{PROJ_DIR}/experiments/spatial/run_compare.py',
     '--config', tmp.name, '--from-models', MODELS_PT],
)
os.unlink(tmp.name)

if result.returncode == 0:
    print('\n✓ Re-extraction OK')
    # Show the newly generated run
    import glob
    runs = sorted(glob.glob(f'{DRIVE_RESULTS}/compare_*'))
    NEW_RUN_DIR = runs[-1]
    print('New run dir:', NEW_RUN_DIR)
    import run_compare as rc, importlib; importlib.reload(rc)
    rc.show_plots_from_dir(NEW_RUN_DIR)
else:
    print('✗ Re-extraction FAILED (see output above)')
